In [2]:
import torch
import torch.nn as nn

In [3]:
class Layer1(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Conv2d(3, 80, 6, 2, 2 )

    def forward(self, x):
        return self.layer1(x)

In [4]:
x = torch.rand(3, 640, 640)
model = Layer1()
out = model(x)
print(out.shape)

torch.Size([80, 320, 320])


In [5]:
### Import necessary libraries and dependencies
import torch
import torch.nn as nn
import math

class GhostModule(nn.Module):
    def __init__(self, inp, oup, kernel_size=1, ratio=2, dw_size=3, stride=1, relu=True):
        super(GhostModule, self).__init__()
        self.oup = oup
        ### Compute channels for both primary and secondary based on ratio (s)
        init_channels = math.ceil(oup / ratio)
        new_channels = init_channels*(ratio-1)

		### Primary standard convolution + BN + ReLU
        self.primary_conv = nn.Sequential(
            nn.Conv2d(inp, init_channels, kernel_size, stride, kernel_size//2, bias=False),
            nn.BatchNorm2d(init_channels),
            nn.ReLU(inplace=True) if relu else nn.Sequential(),
        )

		### Secondary depthwise convolution + BN + ReLU
        self.cheap_operation = nn.Sequential(
            nn.Conv2d(init_channels, new_channels, dw_size, 1, dw_size//2, groups=init_channels, bias=False),
            nn.BatchNorm2d(new_channels),
            nn.ReLU(inplace=True) if relu else nn.Sequential(),
        )

    def forward(self, x):
        x1 = self.primary_conv(x)
        x2 = self.cheap_operation(x1)
        ### Stack
        out = torch.cat([x1,x2], dim=1)
        return out[:,:self.oup,:,:]


In [6]:
GhostModel = GhostModule(inp=80, oup=160, kernel_size=3, stride=2, ratio=2)
out_layer1 = torch.rand(1 , 80, 320, 320)

layer2_out = GhostModel(out_layer1)

layer2_out.shape

torch.Size([1, 160, 160, 160])

In [7]:
import torch
import torch.nn as nn


### DWConv + BN + ReLU
def depthwise_conv(inp, oup, kernel_size=3, stride=1, relu=False):
    return nn.Sequential(
        nn.Conv2d(inp, oup, kernel_size, stride,
                  kernel_size // 2, groups=inp, bias=False),
        nn.BatchNorm2d(oup),
        nn.ReLU(inplace=True) if relu else nn.Sequential(),
    )


### Ghost Bottleneck (NO SE)
class GhostBottleneck(nn.Module):
    def __init__(self, inp, hidden_dim, oup, kernel_size, stride):
        super(GhostBottleneck, self).__init__()
        assert stride in [1, 2]

        # Main branch
        self.conv = nn.Sequential(
            # Pointwise Ghost (expansion)
            GhostModule(inp, hidden_dim, kernel_size=1, relu=True),

            # Depthwise Conv (only if stride=2)
            depthwise_conv(hidden_dim, hidden_dim, kernel_size, stride, relu=False)
            if stride == 2 else nn.Sequential(),

            # Pointwise Ghost (projection)
            GhostModule(hidden_dim, oup, kernel_size=1, relu=False),
        )

        # Shortcut branch
        if stride == 1 and inp == oup:
            self.shortcut = nn.Sequential()  # identity
        else:
            self.shortcut = nn.Sequential(
                depthwise_conv(inp, inp, kernel_size, stride, relu=False),
                nn.Conv2d(inp, oup, kernel_size=1, stride=1,
                          padding=0, bias=False),
                nn.BatchNorm2d(oup),
            )

    def forward(self, x):
        return self.conv(x) + self.shortcut(x)

In [8]:
import torch

# Example input tensor
x = torch.randn(1, 32, 64, 64)   # (B, C, H, W)

# Instantiate GhostBottleneck
block = GhostBottleneck(
    inp=32,
    hidden_dim=64,
    oup=32,
    kernel_size=3,
    stride=1
)

# Forward pass
y = block(x)

print("Input shape :", x.shape)
print("Output shape:", y.shape)

Input shape : torch.Size([1, 32, 64, 64])
Output shape: torch.Size([1, 32, 64, 64])


In [9]:
class Conv(nn.Module):
    def __init__(self, inp, oup, k=1, s=1, p=None, act=True):
        super().__init__()
        if p is None:
            p = k // 2
        self.conv = nn.Conv2d(inp, oup, k, s, p, bias=False)
        self.bn = nn.BatchNorm2d(oup)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

In [10]:
class C3Ghost(nn.Module):
    """
    CSP-style C3 module using GhostBottleneck
    Replacement for YOLOv8 C2f
    """
    def __init__(self, c1, c2, n=1, shortcut=True, e=0.5):
        """
        c1: input channels
        c2: output channels
        n : number of GhostBottlenecks
        e : expansion ratio
        """
        super().__init__()
        c_ = int(c2 * e)  # hidden channels

        # Split paths
        self.cv1 = Conv(c1, c_, k=1, s=1)
        self.cv2 = Conv(c1, c_, k=1, s=1)

        # Ghost bottleneck stack
        self.m = nn.Sequential(*[
            GhostBottleneck(
                inp=c_,
                hidden_dim=c_,
                oup=c_,
                kernel_size=3,
                stride=1
            )
            for _ in range(n)
        ])

        # Fuse
        self.cv3 = Conv(2 * c_, c2, k=1, s=1)

    def forward(self, x):
        y1 = self.m(self.cv1(x))
        y2 = self.cv2(x)
        return self.cv3(torch.cat((y1, y2), dim=1))

In [11]:
import torch
import torch.nn as nn

class GhostBackbone(nn.Module):
    def __init__(self):
        super().__init__()

        # P1: 640x640x3 → 320x320x80
        # FIX: k=6, s=2, p=2 (manual override)
        self.p1 = Conv(3, 80, k=6, s=2, p=2)

        # P2: 320x320x80 → 160x160x160
        self.p2 = GhostModule(80, 160, kernel_size=3, stride=2)

        # C3Ghost (n=3): 160x160x160 → 160x160x160
        self.c3_1 = C3Ghost(160, 160, n=3)

        # P3: 160x160x160 → 80x80x320
        self.p3 = GhostModule(160, 320, kernel_size=3, stride=2)

        # C3Ghost (n=6): 80x80x320 → 80x80x320
        self.c3_2 = C3Ghost(320, 320, n=6)

        # P5: 80x80x320 → 40x40x640
        self.p5 = GhostModule(320, 640, kernel_size=3, stride=2)

        # C3Ghost (n=6): 40x40x640 → 40x40x640
        self.c3_3 = C3Ghost(640, 640, n=6)

        # P7: 40x40x640 → 20x20x640
        self.p7 = GhostModule(640, 640, kernel_size=3, stride=2)

        # C3Ghost (n=3): 20x20x640 → 20x20x640
        self.c3_4 = C3Ghost(640, 640, n=3)

        # GhostBottleneck (no downsample): 20x20x640 → 20x20x640
        self.gb = GhostBottleneck(
            inp=640,
            hidden_dim=640,
            oup=640,
            kernel_size=3,
            stride=1
        )

    def forward(self, x):
        x = self.p1(x)
        x = self.p2(x)
        x = self.c3_1(x)
        x = self.p3(x)
        x = self.c3_2(x)
        x = self.p5(x)
        x = self.c3_3(x)
        x = self.p7(x)
        x = self.c3_4(x)
        x = self.gb(x)
        return x

In [12]:
import torch

# Instantiate model
model = GhostBackbone()

# Dummy input (batch_size=1, 3-channel RGB, 640x640)
x = torch.randn(1, 3, 640, 640)

# Forward pass
with torch.no_grad():
    y = model(x)

# Output shape
print("Final output shape:", y.shape)

Final output shape: torch.Size([1, 640, 20, 20])


In [13]:
import torch
import torch.nn as nn

class GhostBackbone(nn.Module):
    def __init__(self):
        super().__init__()

        # P1: 640x640x3 → 320x320x80
        # FIX: k=6, s=2, p=2 (manual override)
        self.p1 = Conv(3, 80, k=6, s=2, p=2)

        # P2: 320x320x80 → 160x160x160
        self.p2 = GhostModule(80, 160, kernel_size=3, stride=2)

        # C3Ghost (n=3): 160x160x160 → 160x160x160
        self.c3_1 = C3Ghost(160, 160, n=3)

        # P3: 160x160x160 → 80x80x320
        self.p3 = GhostModule(160, 192, kernel_size=3, stride=2)

        # C3Ghost (n=6): 80x80x320 → 80x80x320
        self.c3_2 = C3Ghost(192,192, n=6)

        # P5: 80x80x320 → 40x40x640
        self.p5 = GhostModule(192,384, kernel_size=3, stride=2)

        # C3Ghost (n=6): 40x40x640 → 40x40x640
        self.c3_3 = C3Ghost(384, 384, n=6)

        # P7: 40x40x640 → 20x20x640
        self.p7 = GhostModule(384, 576, kernel_size=3, stride=2)

        # C3Ghost (n=3): 20x20x640 → 20x20x640
        self.c3_4 = C3Ghost(576, 576, n=3)

        # GhostBottleneck (no downsample): 20x20x640 → 20x20x640
        self.gb = GhostBottleneck(
            inp=576,
            hidden_dim=576,
            oup=576,
            kernel_size=3,
            stride=1
        )

    def forward(self, x):
        x = self.p1(x)
        x = self.p2(x)
        x = self.c3_1(x)

        x = self.p3(x)
        p3 = self.c3_2(x)     # 80×80×320

        x = self.p5(p3)
        p4 = self.c3_3(x)     # 40×40×640

        x = self.p7(p4)
        x = self.c3_4(x)
        p5 = self.gb(x)       # 20×20×640

        return p3, p4, p5

In [15]:
import torch

# Instantiate model
model = GhostBackbone()

# Dummy input (batch_size=1, 3-channel RGB, 640x640)
x = torch.randn(1, 3, 640, 640)

# Forward pass
with torch.no_grad():
    p3, p4, p5 = model(x)

# Output shape
print("Final output shape:", p3.shape)
print("Final output shape:", p4.shape)
print("Final output shape:", p5.shape)

Final output shape: torch.Size([1, 192, 80, 80])
Final output shape: torch.Size([1, 384, 40, 40])
Final output shape: torch.Size([1, 576, 20, 20])
